[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Jibby2k1/SPS_Curriculum/blob/main/Intro_DSP/Statistical_Signal_Processing.ipynb)


**Content Produced by UF Signal Processing Society**

**Authors: Raul Valle & Contributors**

# Statistical Signal Processing

Real signals are random. This workshop supplies the theory the [adaptive filtering](../Intro_Time_Series/README.md) notebooks borrowed on credit: random processes and stationarity, spectral estimation done honestly, the Wiener filter derived, and matched filters & detection — the statistics of pulling signals out of noise.

## 1. Pre-requisites

- [Random Variables](../Intro_Math/Analysis/Random_Variables.ipynb) & [Independence](../Intro_Math/Analysis/Independence.ipynb).
- [Foundations of Signal Processing 1](./Foundations_of_Signal_Processing_1.ipynb) (DFT, convolution).
- [Estimation Theory](../Intro_Math/Estimation_Theory/Estimation_Theory.ipynb) helps for Session 4.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy import signal as sig
rng = np.random.default_rng(0)

---
### 🕐 Session 1 of 4 — *Random Processes & Stationarity* (~35 min)
**Goal:** treat a signal as a family of random variables; define autocorrelation and WSS.
**Builds on:** [Random Variables](../Intro_Math/Analysis/Random_Variables.ipynb). &nbsp; **Feeds into:** Session 2 (spectral estimation).

---

## 2. Random Processes

💡 **Intuition.** A random process is a random variable *per time index* — one experiment produces a whole waveform (a *realization*). The process's personality lives in its joint statistics, but for signal processing two numbers usually suffice: the mean $\mu[n]$ and the **autocorrelation** $r[n, m] = E[x[n]x[m]]$ — how much the process remembers itself across time. **Wide-sense stationary (WSS)** means those two don't care about absolute time: $\mu$ constant, $r$ depends only on the lag $k = n - m$. Stationarity is what lets one long recording stand in for the whole ensemble (ergodicity — the [LLN](../Intro_Math/Analysis/Independence.ipynb) applied along time).

In [ ]:
# Three processes, three memories: white, AR(1) smooth, AR(1) alternating

# YOUR CODE HERE


**What just happened.** Three processes, and the two panels show the same information twice. On the left, one realisation each: white noise looks like static, $a = 0.9$ looks smooth and wandering, $a = -0.9$ looks violently jittery. On the right, the autocorrelations: white noise is a spike at lag 0 and nothing after, $a = 0.9$ decays smoothly and positively, and $a = -0.9$ **alternates in sign** as it decays.

**The two AR processes are the instructive pair.** They have identical memory *strength* — $|a| = 0.9$ in both — and look nothing alike. The positive one remembers with a plus sign, so each sample resembles the last and the waveform is smooth. The negative one remembers with a minus sign, so each sample tends to be the opposite of the last, and the result looks like noise despite being just as strongly correlated. **Correlation strength and visual smoothness are different properties**, and the sign of the memory decides which you see. Anyone who equates "correlated" with "smooth" gets this wrong.

That also previews Session 2. Positive memory means energy concentrated at *low* frequency; alternating memory means energy at *high* frequency. The autocorrelation and the PSD are two views of the same object — which is precisely the Wiener–Khinchin theorem.

**Note the luxury this cell is quietly using.** `R = 400` realisations, and `acf_ensemble` averages across them — a genuine *ensemble* average, exactly as the definition $E[x[n]x[m]]$ demands. No real experiment gets 400 independent runs of the same process. In practice you have one recording and you replace the ensemble average with a **time** average, which is legal only under **ergodicity**: the assumption that one long realisation explores the same statistics the ensemble would. That is the [law of large numbers](../Intro_Math/Analysis/Independence.ipynb) applied along time instead of across trials, and every estimator in the rest of this workshop depends on it.

**And WSS is an assumption, not an observation.** Stationarity says the mean and autocorrelation do not depend on absolute time. These synthetic processes satisfy it by construction. Speech does not — its statistics change every phoneme. Nor does a radar return, or music, or an ECG. WSS is imposed to make the theory work, and much of the rest of the DSP track exists to handle what happens when it fails: [time–frequency analysis](./Time_Frequency_2.ipynb) for signals whose spectrum moves, [cyclostationarity](./Cyclostationary_HOS.ipynb) for signals whose statistics repeat periodically.

---
### 🕐 Session 2 of 4 — *The Power Spectral Density* (~40 min)
**Goal:** define the PSD; learn why the raw periodogram lies and how Welch fixes it.
**Builds on:** Session 1. &nbsp; **Feeds into:** Session 3 (Wiener).

---

## 3. The PSD

💡 **Intuition.** The PSD is the autocorrelation's Fourier transform (Wiener–Khinchin): it says how the process's *power* is distributed over frequency — the ensemble version of the spectrum. The trap: the **raw periodogram** $|X(\omega)|^2/N$ is an *inconsistent* estimator — its variance never shrinks, no matter how much data you record, because each frequency bin is essentially one squared Gaussian (~1 degree of freedom, χ²₂-fluctuating forever). **Welch's fix**: chop into segments, periodogram each, *average* — trading resolution for the variance decay the [LLN](../Intro_Math/Analysis/Independence.ipynb) provides.

In [ ]:
# One AR(2) process, its TRUE spectrum, and two estimates from the SAME data

# YOUR CODE HERE


**What just happened.** Both estimates come from the **same 32768 samples**. The Welch curve sits on the true PSD; the raw periodogram fuzzes wildly around it, spanning more than an order of magnitude at neighbouring frequencies. More data did not fix the periodogram — and it never will.

**Why the periodogram is inconsistent, precisely.** At each frequency, the DFT coefficient is essentially one complex Gaussian, so $|X(\omega)|^2$ is a $\chi^2_2$ random variable: **two degrees of freedom, independent of $N$**. For $\chi^2_2$ the standard deviation equals the mean, so the relative error is **100%**, forever. Collecting more samples does not average anything at a given frequency; it merely produces *more frequency bins*, each just as noisy as before. This is the rare case where an estimator does not improve with data, and it is worth sitting with — "more data" and "more averaging" are not the same operation.

**Welch fixes the diagnosis rather than the symptom.** If the problem is one degree of freedom per bin, obtain more: chop into $K$ segments, periodogram each, average. Each bin becomes $\chi^2_{2K}$ and the relative error falls as $1/\sqrt{K}$. Here $N = 32768$ with 512-sample segments at 50% overlap gives roughly **127 segments**, so relative error drops from 100% to about **9%**. That factor of eleven is the entire visible difference between the two traces, and it is the [law of large numbers](../Intro_Math/Analysis/Independence.ipynb) doing exactly what it always does — once you arrange for something to actually be averaged.

**And the price is resolution.** Full-length bins would be $1/32768$ wide; 512-sample segments make them 64× coarser. Two spectral peaks closer together than the segment bandwidth will merge into one and no amount of averaging separates them. So `nperseg` is a **variance-versus-resolution dial**, not a performance setting: small segments give a smooth, blurry estimate; large segments give a sharp, noisy one. Try 64 and 4096 to see one knob move both properties in opposite directions.

**Which estimator is "right" depends on the question.** The periodogram remains perfectly serviceable for *locating* a strong narrow tone — a peak stands out even amid 100% fuzz — and is useless for *estimating a spectral level*, where you need the value rather than the position. Welch is the reverse. Notice too that the fuzz straddles the true curve rather than sitting off to one side: the periodogram is unbiased and merely high-variance, which is why the eye can still read its shape.

That distinction matters immediately: Session 3's Wiener filter is built entirely from PSD *values*, so it needs Welch. A periodogram-based Wiener filter would inherit 100% error in every gain it computes.

---
### 🕐 Session 3 of 4 — *The Wiener Filter, Derived* (~35 min)
**Goal:** solve the optimal linear filtering problem the adaptive filters approximate.
**Builds on:** Session 2; [Linear Algebra](../Intro_Math/Linear_Algebra/Linear_Algebra.ipynb) S2. &nbsp; **Feeds into:** Session 4 (detection).

---

## 4. Optimal Linear Filtering

Problem: estimate desired $d[n]$ from observations $x[n]$ using an FIR filter $\hat{d} = \mathbf{w}^T \mathbf{x}[n]$, minimizing $E[e^2]$.

Setting the gradient to zero (the [matrix calculus](../Intro_Math/Linear_Algebra/Linear_Algebra.ipynb) you've done) gives the **Wiener–Hopf equations**
$$R \mathbf{w}_o = \mathbf{p}, \qquad R = E[\mathbf{x}\mathbf{x}^T], \;\; \mathbf{p} = E[d \, \mathbf{x}],$$
— a projection ([orthogonality principle](../Intro_Math/Hilbert_Spaces/Hilbert_Spaces.ipynb): the optimal error is orthogonal to every observation). [LMS/APA](../Intro_Time_Series/Intro_AdFilt_APA.ipynb) chase this solution without knowing $R, \mathbf{p}$; here we *compute* it.

💡 **Intuition.** In the frequency domain the noncausal solution is transparent: $W(\omega) = \frac{S_d(\omega)}{S_d(\omega) + S_v(\omega)}$ for signal-plus-noise — a **per-frequency trust dial** (compare the Kalman gain!). Where signal dominates, pass ≈ 1; where noise dominates, squash ≈ 0. Optimal filtering is spectral triage.

In [ ]:
# Wiener denoising, built from PSDs alone
# apply as zero-phase frequency-domain filter (block processing)

# YOUR CODE HERE


**What just happened.** Input SNR **10.1 dB**, output SNR **13.3 dB** — a 3.2 dB gain, computed from **PSDs alone**. The filter never touched `d`'s samples; `W = S_d / (S_d + 1.0)` was built from a spectral estimate and a known noise level, and that was enough.

**The formula is a per-frequency trust dial.** $W(\omega) = S_d/(S_d + S_v)$ approaches 1 where the signal dominates and 0 where noise dominates, sliding smoothly between. Optimal filtering turns out to be *spectral triage*: at each frequency, weight by how much of what you are hearing is actually signal. Note the structural identity with the Kalman gain from [the Kalman workshop](../Intro_Time_Series/Intro_AdFilt_KF.ipynb) — signal variance over total variance — one indexed by frequency, the other by time. The same "trust each source in proportion to its certainty" logic runs through both.

**Now be honest about 3.2 dB, because it is a modest number and the reason matters.** The desired signal is AR(1) with $a = 0.95$, so its power concentrates at low frequency — but it is *not* band-limited. Its spectrum has tails that overlap the white noise across the entire band. Wiener can only exploit the *difference* in spectral shape, so wherever signal and noise genuinely coexist, even the optimal linear filter must pass some noise and attenuate some signal. **The achievable gain is a property of the two spectra, not of the filter.** Give the signal a narrower band, or give the noise a spectrum that avoids the signal band, and the same filter delivers far more. Nothing about the algorithm would change.

That is worth stating because it inverts the usual reading. A disappointing result here would not mean "use a better filter" — this *is* the best linear filter, provably. It means the problem itself has limited headroom, which is a much more useful diagnosis.

**Why the derivation matters as much as the result.** Setting $\partial E[e^2]/\partial\mathbf{w} = 0$ gives the Wiener–Hopf equations $R\mathbf{w}_o = \mathbf{p}$, and that condition says precisely $E[e\cdot\mathbf{x}] = 0$: **the optimal error is orthogonal to every observation**. So the Wiener filter is a *projection* of the desired signal onto the span of the data — the [Hilbert space projection theorem](../Intro_Math/Hilbert_Spaces/Hilbert_Spaces.ipynb) appearing in yet another costume. Intuitively, orthogonality means no further linear function of the observations could reduce the error: everything linearly extractable has been extracted.

**And this closes a debt.** [LMS, NLMS, and APA](../Intro_Time_Series/Intro_AdFilt_APA.ipynb) were all introduced as chasing "the Wiener solution" without one ever being computed, because $R$ and $\mathbf{p}$ are unknown in practice. Here we computed it. The adaptive filters are the online, statistics-free approximations to this cell; this cell is what they are approximating.

One caveat on the implementation: `W` is applied as a zero-phase frequency-domain multiply over the whole block, which is the *non-causal* Wiener filter. It uses future samples, so it is fine for offline processing and unavailable for real-time work. The causal version requires spectral factorisation and performs somewhat worse — a real distinction worth knowing before deploying this.

---
### 🕐 Session 4 of 4 — *Matched Filters & Detection* (~40 min)
**Goal:** detect a known pulse in noise optimally; meet ROC curves and the Neyman–Pearson view.
**Builds on:** Session 3; [Estimation Theory](../Intro_Math/Estimation_Theory/Estimation_Theory.ipynb).

---

## 5. Detection

💡 **Intuition.** Estimation asks 'what is $\theta$?'; detection asks '**is it there at all?**' For a known pulse in white Gaussian noise, the optimal detector correlates the data against the pulse — the **matched filter**, which is just the [Cauchy–Schwarz](../Intro_Math/Hilbert_Spaces/Hilbert_Spaces.ipynb) statement that correlation against a template is maximized by the template itself. It maximizes SNR at the decision instant; radar, sonar, GPS, and your Wi-Fi preamble sync all run on it.

**Neyman–Pearson framing.** Choose the threshold to fix the false-alarm rate $P_{FA}$; the likelihood-ratio test (here: matched filter output vs threshold) then maximizes detection probability $P_D$. Sweeping the threshold traces the **ROC curve** — the universal report card of any detector, from radar to medical tests to spam filters.

In [ ]:
# Matched filter vs naive energy detector, at SNR where it matters

# YOUR CODE HERE


**What just happened.** Two ROC curves from the same 4000 trials of the same data, and the matched filter's sits **entirely above** the energy detector's. That is a stronger statement than "it scores better": a curve that dominates everywhere means the matched filter wins at *every* operating point, whatever false-alarm rate you choose. The comparison is threshold-free, which is exactly what the ROC is for.

**Why the energy detector loses.** It computes $\sum x^2$ — "is there more energy than usual?" — and in doing so it *discards the template completely*. It would respond identically to noise of the same total power, or to a pulse of entirely the wrong shape. The matched filter computes $x \cdot h$, asking "is there energy **shaped like my pulse**?", and that extra question is worth the whole gap between the curves.

The general principle is one this curriculum keeps returning to: **what you know about the signal is what buys performance**. It is the same lesson as L1-versus-L2 in [Compressed Sensing](./Compressed_Sensing.ipynb) — the prior, not the data, is doing the work.

**And the matched filter is optimal for a provable reason.** Cauchy–Schwarz says $|\langle x, h\rangle| \le \|x\|\,\|h\|$ with equality precisely when $h \propto x$. So among all linear statistics of a given norm, correlating against the template maximises the output at the decision instant — this is the [Hilbert space](../Intro_Math/Hilbert_Spaces/Hilbert_Spaces.ipynb) inequality doing detection theory. Not "a good idea that works well," but the best possible linear statistic, with a one-line proof.

**Read the ROC properly.** The diagonal is a coin flip — a detector on that line has no information. Up and to the left is better. Sweeping the threshold moves you *along* a curve, trading false alarms against missed detections, but never off it: the curve is the detector's capability, and the threshold is only where on it you choose to sit. That choice is a **cost judgement**, not a statistical one. A radar hunting aircraft, a cancer screening test, and a spam filter should sit at wildly different points on their curves, because the relative cost of a false alarm versus a miss differs in each. Neyman–Pearson formalises this: fix $P_{FA}$ at what you can tolerate, then maximise $P_D$ subject to it.

**The assumptions are narrow, and worth stating.** The matched filter is optimal for a *known* pulse, at a *known* arrival time, in *white Gaussian* noise. Relax any of the three and it degrades: unknown timing needs a sliding correlation or a filter bank — which is precisely what radar [pulse compression](./Radar_Signal_Processing.ipynb) is; coloured noise requires whitening first, then matching; an unknown pulse shape needs a generalised likelihood ratio test. The energy detector, for all its weakness here, needs *none* of that knowledge, which is why it survives in practice and why [cyclostationary detection](./Cyclostationary_HOS.ipynb) exists for the middle ground where you know the signal's structure but not its details.

## 6. Conclusion

WSS + ergodicity let one recording speak for the ensemble; Welch buys consistent spectra with the LLN; Wiener filtering is spectral triage and the target all adaptive filters chase; matched filtering + Neyman–Pearson is optimal 'is it there?'. This is the statistical spine of practical DSP.

---
## Where next

- [Adaptive Filtering](../Intro_Time_Series/Intro_AdFilt_APA.ipynb) — Wiener pursued online.
- [Array Processing](./Array_Processing.ipynb) — these tools across space, not just time.
- [Digital Communications](./Digital_Communications.ipynb) — matched filters earning rent every symbol.